# 02 · Configurações 1 e 2: modelo sem contexto × RAG simples
Mesmo modelo (`Qwen2.5-1.5B-Instruct`, rodando localmente na GPU do Colab), mesmo pedido de JSON.
A única diferença é o contexto recuperado da base.

In [1]:
# Preparação do ambiente (Colab ou local)
import os, sys, subprocess
REPO_URL = "https://github.com/giuliano10/GS-GENENERATIVE-AI.git"
SUBPASTA = "assistente-acessibilidade-calcadas"   # pasta do projeto dentro do repositório

if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    else:   # mesma sessão de outro notebook: atualiza o código
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
    os.chdir(os.path.join("/content/repo", SUBPASTA))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("Falha no pip install (detalhes acima)")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())

Pasta do projeto: /content/repo/assistente-acessibilidade-calcadas


In [2]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (use Ambiente de execução > Alterar tipo > T4 GPU)")

GPU: Tesla T4


In [3]:
from src.base_conhecimento import carregar_trechos
from src.avaliacao import carregar_jsonl
from src.busca import Buscador, carregar_limiar
from src.config import ARQ_TESTE
from src.gerador import carregar_modelo
from src.experimento import rodar
trechos = carregar_trechos(); teste = carregar_jsonl(ARQ_TESTE)
buscador = Buscador(trechos, modo="hibrido"); limiar = carregar_limiar()
modelo = carregar_modelo(com_lora=False)
print(modelo.nome, "| limiar", limiar)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

... (10 outputs left out)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-1.5B-Instruct | limiar 0.8


## Configuração 1 — sem contexto

In [4]:
sem_ctx = rodar("sem_contexto", teste, modelo)
for s in sem_ctx[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

[sem_contexto] 10/60
[sem_contexto] 20/60
[sem_contexto] 30/60
[sem_contexto] 40/60
[sem_contexto] 50/60
[sem_contexto] 60/60
Qual a largura mínima da faixa livre da calçada em São Paulo? 
 -> {
  "resposta": "Deve ser igual ou maior à metade do diâmetro máximo dos veículos.",
  "fontes": ["Lei nº 13.428/2016"],
  "recusa": false
} 

Qual a inclinação transversal máxima permitida na faixa livre da calçada? 
 -> {
  "resposta": "A inclinação transversal máxima permitida na faixa livre da calçada é de 10%.",
  "fontes": ["Lei Municipal nº 23.465/2017"],
  "recusa": false
} 

Qual a largura mínima da faixa de serviço da calçada? 
 -> {
  "resposta": "Deve ser igual ou maior à metade do diâmetro máximo dos veículos.",
  "fontes": ["Lei nº 13.709/2018"],
  "recusa": false
} 



## Configuração 2 — RAG simples

In [5]:
rag = rodar("rag", teste, modelo, buscador, limiar)
for s in rag[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

[rag] 10/60
[rag] 20/60
[rag] 30/60
[rag] 40/60
[rag] 50/60
[rag] 60/60
Qual a largura mínima da faixa livre da calçada em São Paulo? 
 -> {"resposta": "Não consta na base de conhecimento.", "fontes": [], "recusa": true} 

Qual a inclinação transversal máxima permitida na faixa livre da calçada? 
 -> {"resposta": "Não consta na base de conhecimento.", "fontes": [], "recusa": true} 

Qual a largura mínima da faixa de serviço da calçada? 
 -> {"resposta": "Não consta na base de conhecimento.", "fontes": [], "recusa": true} 



In [6]:
import pandas as pd
from src.avaliacao import corrigir, placar
ids = {t["id"] for t in trechos}; fids = {t["fonte_id"] for t in trechos}
pd.DataFrame([placar([corrigir(c, s, ids, fids) for c, s in zip(teste, saidas)], nome)
              for nome, saidas in [("sem_contexto", sem_ctx), ("rag", rag)]])

,configuracao,n,acuracia_%,acerto_consta_%,recusa_correta_nao_consta_%,alucinacao_%,recusa_indevida_%,json_valido_%,citacao_correta_%,citacao_fora_contexto_%,latencia_media_s
0,sem_contexto,60,11.7,11.1,13.3,88.3,0.0,96.7,0.0,NaN,2.78
1,rag,60,25.0,0.0,100.0,0.0,100.0,100.0,NaN,0.0,0.01
